# 4 MNIST dataset
In this section, you will load and explore the MNIST dataset, a collection of handwritten digits from 0 to 9 that you have already encountered in previous laboratories.

- What it is:
    - A dataset of handwritten digits from 0 to 9.
- Images:
    - 28 × 28 pixels
    - grayscale (1 channel, values typically 0–255 or 0–1 if normalized)
    - Each image contains one digit, roughly centered.
- Size:
    - 60,000 images for training
    - 10,000 images for test
    - So total: 70,000 labeled images.
- Labels:
    - Integer in {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
    - So it’s a 10-class classification problem.

PyTorch provides a ready-to-use interface through **torchvision.datasets.MNIST**, which automatically downloads and organizes the data.

Use this:

In [ ]:
import matplotlib.pyplot as plt

import torch
# import torchvision.transforms
from torchvision import datasets, transforms

import numpy as np

In [ ]:
train_dataset = datasets.MNIST(root="data", train=True, download=True)
test_dataset = datasets.MNIST(root="data", train=False, download=True)

- Dataset size and Accessing samples: Check the number of samples contained in the training and test datasets by printing their lengths. Each element i of the dataset can be accessed as a pair (image, label) simply using train_dataset[i]. Retrieve one element and inspect its structure and the shape of the *(mistake: tensors, they're not tensors yet but you need to convert them inot tensors later)*.

- Visual inspection: Plot a small grid of sample images together with their corresponding labels to confirm that you can correctly access both the data and its annotations. Remember that MNIST images are stored as gray scale tensors, so you should display them using the parameter cmap=’gray’ when calling **imshow()** to avoid color distortions.

In [ ]:
print(train_dataset)
# wtf is this?

print(type(train_dataset))
# weird ass class --> <class 'torchvision.datasets.mnist.MNIST'>
# This object is like taking the original data (x) and corresponding labels (y) and doing TensorDataset(x,y), returns the dataset but divided in tuples (x_i, y_i) --> (data, corresponding labels)

print('\nThe training dataset has length')
print(len(train_dataset))

print('\nThe test dataset has length')
print(len(test_dataset))

for image,label in train_dataset:
    print(f"The image:\t{image}")
    print(type(image))
    print(f"Has label:\t{label}")
    print(type(label))
    break

Plot to inspect data

In [ ]:
# plot a sample of images, so some images and their corresponding label
    # use plt.imshow(cmap='gey')
        # imshow(X, cmap=None, norm=None, *, aspect=None, interpolation=None, alpha=None, vmin=None, vmax=None, colorizer=None, ...)
        # Xarray-like or PIL image
    # plot the image and as a title of the plot use the label/class


plt.figure(figsize=(10,10))
for idx in range(6):
    subplot_idx = idx + 1
    image = train_dataset[idx][0]
    label = train_dataset[idx][1]

    plt.subplot(2,3,subplot_idx)
    plt.imshow(image, cmap='gray')
    plt.title(label)

plt.tight_layout()
plt.show()

# I FINALLY MADE A PLOT MYSELF!!!!!!
# plt.figure(figsize=(10,10))
# 	•	This creates the “sheet of paper”.
# 	•	figsize=(10,10) = 10×10 inches

# for idx in range(6):
#     subplot_idx = idx + 1
# 	•	range(6) → idx goes: 0,1,2,3,4,5
# 	•	subplot_idx goes: 1,2,3,4,5,6
# You need 1..6 because plt.subplot uses 1-based indexing for the position.

# plt.subplot(2,3,subplot_idx)
# = plt.subplot(n_rows, n_cols, index)
# YOU'RE BUILDING A 2X3 MATRIX / GRID AND YOU'RE DRAWING INSIDE THE INDEX=SUBPLOT_IDX CELL, SO:
# 	•	subplot(2,3,1) → row 1, col 1
# 	•	subplot(2,3,2) → row 1, col 2
# 	•	subplot(2,3,3) → row 1, col 3
# 	•	subplot(2,3,4) → row 2, col 1
# 	•	subplot(2,3,5) → row 2, col 2
# 	•	subplot(2,3,6) → row 2, col 3

# plt.imshow(image, cmap='gray')
# plt.title(label)
# 	•	imshow(image, cmap='gray'): display the 28×28 image in grayscale.
# 	•	plt.title(label): put the digit (the label) as the title on top of that subplot.

# 	•	Every time you call plt.subplot(...), it selects a cell.
# 	•	Every time you call plt.imshow(...), it draws in the currently active cell.
# 	•	The loop just keeps changing the active cell and drawing there.
# 	•	Nothing is actually shown until plt.show() is called at the end.

# plt.tight_layout() just rearranges spacing so titles/axes don’t overlap.


### 4.1 Preprocessing and transforms
Before training a model, image data must often be transformed into a suitable numerical format and standardized for more stable learning. In PyTorch, this process is handled by the **torchvision.transforms** module, which provides a collection of ready-to-use operations that can be composed together in a pipeline.  

> Pipeline? Yes, we are building a preprocessing pipeline, we'll use **transforms.Compose** which is the exact same thing as make_pipeline(...) in scikit-learn.

- Conversion to tensors: Raw MNIST images are loaded as PIL images. The **ToTensor()** transform converts them into PyTorch tensors of shape 1 × 28 × 28, automatically scaling pixel values from the range[0,255] to[0,1]. The first dimension, 1, represents the number of “channels” of the image: since the images are grayscale, there is a single color channel. Colored images are typically represented using 3 channels (Red, Green, Blue, or RGB).

In [ ]:
# convert PIL images to toch.tensors: first let's see what it does and then perform it on all data

# build the transformer
transformer_img = transforms.ToTensor()

for image, label in train_dataset:
    print(image)
    image = transformer_img(image)
    print(image)
    print('\nINFO:\n')
    print(type(image))
    print(image.shape)
    print(image.min(), image.max())
    print(label)
    break
# <class 'torch.Tensor'>
# torch.Size([1, 28, 28])
# tensor(0.) tensor(1.)
# 5

In [ ]:
# convert PIL images to toch.tensors
    # updating problem: I need to rebuild the data strcture because otherwise I conver the image and label to tensors on the fly but cannot assign it to train_dataset

transformer_img = transforms.ToTensor()

x_train_noob = []
for image, label in train_dataset:
    image = transformer_img(image)
    label = torch.tensor(label)
    x_train_noob.append([image, label])

print(x_train_noob)

This works, takes a bit but works. That's the quickest, laziest and most noob way to do it. The pro way exploits when you recall dataset.MNIST to specify the transformations

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform  # <--
)

image,label = train_dataset[0]
print(type(image), image.shape)
print(label, type(label))
# IF we need it we'll convert labels into tensors later on in the training loop

Normalization: To make learning more efficient, it is common to normalize the data so that pixel intensities have approximately zero mean and unit variance. This is achieved using the transform:

$x_{\text{norm}} = \frac{x-\mu}{\sigma}$

where µ= 0.1307 and σ= 0.3081 are the empirical mean and std of the MNIST dataset.

In [ ]:
# again, even here I have an updating problem --> pipeline

for img, lab in train_dataset:
    mu = 0.1307
    sigma = 0.3081
    img = (img - mu) / sigma


## LET'S BUILD A PREPROCESSING TRANSFORMATION PIPELINE
> **transforms.Compose**
example:
```python
transform_img = T.Compose([
    to_tensor,
    normalize_mnist
])
```

- Is like writing:
```python
def transform_img(x):
    x = to_tensor(x)
    x = normalize_mnist(x)
    return x
```

In [ ]:
mu = 0.1307
sigma = 0.3081

# build the preprocessing transformation pipeline
img_preprocessing_transformation_pipeline = transforms.Compose([
    transforms.ToTensor(),                              # layer 1: from PIL images --> tensor
    transforms.Normalize(mean=[mu], std=[sigma])            # layer 2: normalize tensor images. They have to be in lists, otherwise that stupid shit doesn't work
])

processed_dataset = []
for img, lab in train_dataset:
    img_proc = img_preprocessing_transformation_pipeline(img)       # the image passes though the pipeline: converted into tensor and then normalized
    lab = torch.tensor(lab)         # I mean, since I'm already here, let's also convert the label into a tensor
    processed_dataset.append((img_proc, lab))


- Data augmentation: Simple transformations, such as small random rotations, can be added to slightly modify the images during training. These are well-known data augmentation techniques, that help the model better generalize and be more robust to spatial variations.

Like this:
```python
from torchvision import transforms
transform = transforms. Compose ([
    transforms. RandomRotation (45),
    ・・・
    # add the transforms to convert to tensor and apply the normalization
    ])
```

#### What does RandomRotation do?
> every time you load an image, randomly spin it a bit.

- Every time you apply it, it randomly rotates the image by an angle.
- With a single number 45, the angle is sampled uniformly between -45° and +45°.
- So one call might rotate by +12°, the next by -30°, etc.
- It applies to each image independently and randomly every time you call it.

> FUCK YOU MEAN ROTATE?????

Rotation = we change the coordinates of the pixels: each pixel that was at position (row, col) gets moved to a new position (row', col') according to rotation around the center by some angle θ.  
**So in this way the final image is rotated too!**


> PERFORM ROTATION AS THE FIRST STEP, **ROTATE PIL IMAGES**, NOT TENSORS

In [ ]:
mu = 0.1307
sigma = 0.3081

# build the preprocessing transformation pipeline
img_preprocessing_transformation_pipeline = transforms.Compose([
    transforms.RandomRotation(45),
    transforms.ToTensor(),                              # layer 1: from PIL images --> tensor
    transforms.Normalize(mean=[mu], std=[sigma])            # layer 2: normalize tensor images. They have to be in lists, otherwise that stupid shit doesn't work
])

processed_dataset = []
for img, lab in train_dataset:
    img_proc = img_preprocessing_transformation_pipeline(img)       # the image passes though the pipeline: converted into tensor and then normalized
    lab = torch.tensor(lab)         # I mean, since I'm already here, let's also convert the label into a tensor
    processed_dataset.append((img_proc, lab))


- Applying a transform to a single image: Retrieve one image from the dataset and apply the defined transform directly to it. This will convert the image into a normalized tensor that can be fed into a model. Verify that the transformed image is now a tensor with shape [1, 28, 28].

In [ ]:
img, lab = train_dataset[0]
img_proc = img_preprocessing_transformation_pipeline(img)

print(type(img_proc))
print(img_proc.shape)
# nice :)

In [ ]:
# try to plot because I want to practice
    # but it's not a PIL image anymore, but imshow works with an array too!
    # it's happy with:
	# •	a 2D array: [height, width] → grayscale image
	# •	or a 3D array: [height, width, number of channels: 3 --> RED, Green, Blue] → RGB

# right now our img has shape (1, 28, 28) --> (number of channels (C), height (H), width (W)) --> (C, H, W)
# BUT for MNIST there’s 1 channel (because grayscale), not 3 like RGB --> drop first dimension OR rearrange the dimensions to have (28, 28, 1), which matlab can understand

plt.figure(figsize=(10,10))
for idx in range(6):
    img = processed_dataset[idx][0] # 3D array with stinky first dimension that we want to drop --> (1, 28, 28) --> DROP DIMENSION IF IT HAS SIZE 1
    img_2D = img.squeeze(0)
    img_2D_array = np.array(img_2D)
    lab = processed_dataset[idx][1]

    subplot_idx = idx + 1
    plt.subplot(2,3,subplot_idx)
    plt.imshow(img_2D_array, cmap='gray')
    plt.title(lab)
plt.tight_layout()
plt.show()

# heheheheh, some pictures are rotated :)


- Applying a transform to an existing dataset: If a dataset was created without a transform, you can still attach one afterwards by assigning it to the transform attribute (train _dataset.transform = transform). From this point on, every time a sample is retrieved, the transform will be applied automatically.

> huh?

Basically for now in the code written:
- You take train_dataset (raw MNIST, PIL images).
- You loop over everything, manually apply the pipeline.
- You build a new list called processed_dataset that contains processed (img_proc, lab).
> “I will preprocess everything now, store it in a new list.”

**WHICH IS FINE!!!**

BUT the professor asks for:
> I will teach the dataset how to preprocess itself on the fly every time I ask for a sample.

```python
train_dataset.transform = transform
```
From that moment on:
- img, lab = train_dataset[i]
- will automatically run: img = transform(img) before giving it to you.

In [ ]:
# IGNORE, IT'S USELESS AND JUST CONFUSING!!!
# this works ONLY for this:
	# •	datasets.MNIST is a class.
	# •	Its author decided that:
	# •	self.transform → is for the image
	# •	self.target_transform → is for the label

# define pipeline
mu = 0.1307
sigma = 0.3081

img_preprocessing_transformation_pipeline = transforms.Compose([
    transforms.RandomRotation(45),              # random rotate the PIL image
    transforms.ToTensor(),                      # PIL -> tensor [1, 28, 28], values in [0, 1]
    transforms.Normalize(mean=[mu], std=[sigma])# (x - mu) / sigma
])

# import dataset WITHOUT TRANSFORMATIONS
train_dataset = datasets.MNIST(root="data", train=True, download=True)
test_dataset = datasets.MNIST(root="data", train=False, download=True)

# explain how to transform the datan inside the dataset
train_dataset.transform = img_preprocessing_transformation_pipeline